# Material yield estimator

## Material yield estimator

In [47]:
"""
Material Yield Estimator
========================
Given a location (lon/lat) and a maximum travel distance (km),
estimate the total available yield (tonnes/year) for each material type
from all 10 km raster pixels within that radius.

Raster specs: EPSG:3035, ~10x10 km pixels, float32, nodata=0
"""

import numpy as np
import rasterio
from rasterio.transform import rowcol
from pyproj import Transformer
from pathlib import Path

# ── Configuration ────────────────────────────────────────────────────────────

YIELD_MAP_DIR = Path("../../data/processed/yield_maps")

MATERIALS = [
    "bark",
    "cellulose",
    "cotton",
    "hempdust",
    "peas",
    "sawdust",
    "seagrass",
]

# ── Core function ─────────────────────────────────────────────────────────────

def estimate_yield(
    lon: float,
    lat: float,
    max_distance_km: float,
    materials: list[str] = MATERIALS,
    yield_map_dir: Path = YIELD_MAP_DIR,
    verbose: bool = True,
) -> dict[str, float]:
    """
    Estimate total material yield within a circular catchment area.

    Parameters
    ----------
    lon : float
        Longitude of the facility/origin point (WGS84 decimal degrees).
    lat : float
        Latitude of the facility/origin point (WGS84 decimal degrees).
    max_distance_km : float
        Maximum travel/collection radius in kilometres.
    materials : list[str]
        Material names to query (must match filenames).
    yield_map_dir : Path
        Directory containing yield GeoTIFFs.
    verbose : bool
        Print a summary table when True.

    Returns
    -------
    dict[str, float]
        {material_name: total_yield_tonnes_per_year}
    """

    # 1. Project query point to EPSG:3035
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)
    x_origin, y_origin = transformer.transform(lon, lat)
    max_dist_m = max_distance_km * 1000

    results = {}

    for material in materials:
        tif_path = yield_map_dir / f"yield_tonnes_perYear_10km_{material}.tif"
        if not tif_path.exists():
            print(f"  [WARNING] File not found, skipping: {tif_path}")
            results[material] = np.nan
            continue

        with rasterio.open(tif_path) as src:
            transform = src.transform
            pixel_width  = abs(transform.a)   # metres (x)
            pixel_height = abs(transform.e)   # metres (y)

            # 2. Compute bounding box of the circle in pixel indices
            col_origin, row_origin = ~transform * (x_origin, y_origin)  # float
            col_origin, row_origin = col_origin, row_origin              # keep floats for distance calc

            # Buffer in pixels (ceiling to be safe)
            col_buf = int(np.ceil(max_dist_m / pixel_width))
            row_buf = int(np.ceil(max_dist_m / pixel_height))

            row_min = max(0,          int(row_origin) - row_buf)
            row_max = min(src.height, int(row_origin) + row_buf + 1)
            col_min = max(0,          int(col_origin) - col_buf)
            col_max = min(src.width,  int(col_origin) + col_buf + 1)

            # 3. Read the windowed subset
            window = rasterio.windows.Window(
                col_off=col_min,
                row_off=row_min,
                width =col_max - col_min,
                height=row_max - row_min,
            )
            data = src.read(1, window=window).astype(np.float32)

            # Replace nodata (0) with NaN
            nodata = src.nodata if src.nodata is not None else 0.0
            data[data == nodata] = np.nan

            # 4. Build pixel-centre coordinate arrays (EPSG:3035) for the window
            rows_idx = np.arange(row_min, row_max)
            cols_idx = np.arange(col_min, col_max)
            cols_2d, rows_2d = np.meshgrid(cols_idx, rows_idx)

            # Pixel centres: top-left corner + 0.5 pixel offset
            x_centres = transform.c + (cols_2d + 0.5) * transform.a
            y_centres = transform.f + (rows_2d + 0.5) * transform.e

            # 5. Euclidean distance in EPSG:3035 (metres) — valid for Europe
            dist = np.sqrt((x_centres - x_origin) ** 2 + (y_centres - y_origin) ** 2)

            # 6. Mask: within radius AND valid data
            mask = (dist <= max_dist_m) & ~np.isnan(data)
            total = float(np.nansum(data[mask]))

        results[material] = total

    if verbose:
        _print_summary(results, lon, lat, max_distance_km)

    return results


# ── Helpers ───────────────────────────────────────────────────────────────────

def _print_summary(results: dict, lon: float, lat: float, radius_km: float) -> None:
    print(f"\n{'='*52}")
    print(f"  Material yield estimate")
    print(f"  Origin : {lat:.4f}°N, {lon:.4f}°E")
    print(f"  Radius : {radius_km:.0f} km")
    print(f"{'='*52}")
    print(f"  {'Material':<14}  {'Yield (t/yr)':>16}")
    print(f"  {'-'*14}  {'-'*16}")
    total_all = 0.0
    for mat, val in results.items():
        if np.isnan(val):
            print(f"  {mat:<14}  {'N/A':>16}")
        else:
            print(f"  {mat:<14}  {val:>16,.1f}")
            total_all += val
    print(f"  {'─'*32}")
    print(f"  {'TOTAL':<14}  {total_all:>16,.1f}")
    print(f"{'='*52}\n")

In [48]:
# Example: Amsterdam, 500 km radius
results = estimate_yield(
    lon=4.9,
    lat=52.37,
    max_distance_km=500,
)


  Material yield estimate
  Origin : 52.3700°N, 4.9000°E
  Radius : 500 km
  Material            Yield (t/yr)
  --------------  ----------------
  bark                   372,555.3
  cellulose            2,234,507.5
  cotton                       0.0
  hempdust                35,960.0
  peas                   771,143.1
  sawdust              1,804,438.8
  seagrass                 1,610.9
  ────────────────────────────────
  TOTAL                5,220,215.6



## Visualisation

In [52]:
"""
Material Yield Visualizer
=========================
Companion to material_yield_estimator.py.

Produces a 2×4 figure:
  [0,0] Pie chart – proportion of materials within the catchment
  [0,1..3], [1,0..3] – one map per material (7 total)

Each map shows:
  • Europe land mask (light grey) derived from the boundary raster – same
    pixel grid as the yield maps so edges align perfectly
  • yield raster outside catchment: medium grey (land visible underneath)
  • yield raster inside catchment: material colour sequential colormap
  • dashed catchment circle in material colour
  • origin star marker
  • per-panel colorbar (visible on data panels, hidden but space-reserved on empty panels)
  • shared legend at the bottom
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.patches import Circle
from matplotlib.lines import Line2D
import rasterio
import rasterio.windows
from pyproj import Transformer
from pathlib import Path

# ── Palette ───────────────────────────────────────────────────────────────────

MATERIAL_COLORS = {
    "bark":      "#7B3F00",   # dark brown
    "cellulose": "#1565C0",   # deep blue
    "cotton":    "#F5C842",   # golden yellow
    "hempdust":  "#AEEA00",   # lime / yellow-green
    "peas":      "#2E7D32",   # dark forest green
    "sawdust":   "#E64A19",   # burnt orange-red
    "seagrass":  "#7B1FA2",   # purple
}

YIELD_MAP_DIR    = Path("../../data/processed/yield_maps")
BOUNDARY_RASTER  = Path("../../data/processed/spatial_data/europe_boundary_10km.tif")

# How many radii to show around the origin (1.0 = tight circle edge,
# 1.8 = shows ~80 % extra context beyond the catchment boundary)
MAP_ZOOM_FACTOR = 1.2

# ── Helpers ───────────────────────────────────────────────────────────────────

def _make_sequential_cmap(hex_color: str, name: str):
    """Near-white → material colour sequential colormap."""
    return mcolors.LinearSegmentedColormap.from_list(
        name, ["#f0f0f0", hex_color], N=256
    )


def _get_window_params(tif_path: Path, x_origin: float, y_origin: float,
                       max_dist_m: float, zoom: float = MAP_ZOOM_FACTOR):
    """
    Row/col window that covers zoom × the catchment radius around the origin.
    Also returns pixel_size and the raster transform.
    """
    with rasterio.open(tif_path) as src:
        t = src.transform
        col_origin = (x_origin - t.c) / t.a
        row_origin = (y_origin - t.f) / t.e
        col_buf = int(np.ceil(max_dist_m * zoom / abs(t.a)))
        row_buf = int(np.ceil(max_dist_m * zoom / abs(t.e)))
        row_min = max(0, int(row_origin) - row_buf)
        row_max = min(src.height, int(row_origin) + row_buf + 1)
        col_min = max(0, int(col_origin) - col_buf)
        col_max = min(src.width,  int(col_origin) + col_buf + 1)
        pixel_size = (abs(t.a) + abs(t.e)) / 2
        transform  = t
    return row_min, row_max, col_min, col_max, pixel_size, transform


def _read_windowed(tif_path: Path, row_min, row_max, col_min, col_max,
                   mask_nodata: bool = True):
    with rasterio.open(tif_path) as src:
        window = rasterio.windows.Window(
            col_off=col_min, row_off=row_min,
            width=col_max - col_min, height=row_max - row_min,
        )
        data = src.read(1, window=window).astype(np.float32)
        if mask_nodata:
            nodata = src.nodata if src.nodata is not None else 0.0
            data[data == nodata] = np.nan
    return data


def _build_pixel_centres(transform, row_min, row_max, col_min, col_max):
    cols_idx = np.arange(col_min, col_max)
    rows_idx = np.arange(row_min, row_max)
    cols_2d, rows_2d = np.meshgrid(cols_idx, rows_idx)
    x = transform.c + (cols_2d + 0.5) * transform.a
    y = transform.f + (rows_2d + 0.5) * transform.e
    return x, y


def _read_land_mask(boundary_path: Path, row_min, row_max, col_min, col_max):
    """
    Read the boundary raster and return a boolean land mask (True = land).
    Assumes land pixels have value > 0 and nodata / sea = 0 or NaN.
    Returns None if the file does not exist.
    """
    if not boundary_path.exists():
        print(f"[WARNING] Boundary raster not found: {boundary_path}")
        return None
    data = _read_windowed(boundary_path, row_min, row_max, col_min, col_max,
                          mask_nodata=False)
    return data > 0


def _draw_land(ax, x, y, land_mask):
    """Draw land background as a flat light-grey layer."""
    if land_mask is None:
        return
    land_val = np.where(land_mask, 1.0, np.nan)
    land_cmap = mcolors.ListedColormap(["#e0ddd8"])   # warm light grey
    ax.pcolormesh(x, y, land_val,
                  cmap=land_cmap, shading="nearest",
                  vmin=0.5, vmax=1.5,
                  rasterized=True, zorder=1)





def _draw_circle_and_origin(ax, x_origin, y_origin, max_dist_m, color):
    circle = Circle((x_origin, y_origin), radius=max_dist_m,
                    fill=False, edgecolor=color, linewidth=1.4,
                    linestyle="--", zorder=6)
    ax.add_patch(circle)
    ax.plot(x_origin, y_origin, marker="*", markersize=9,
            color=color, markeredgecolor="white", markeredgewidth=0.6, zorder=7)


def _format_map_ax(ax, x_lim, y_lim):
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.set_aspect("equal")
    ax.set_facecolor("#c9dff0")   # sea / background blue
    ax.set_xticks([])
    ax.set_yticks([])


# ── Main function ─────────────────────────────────────────────────────────────

def visualize_yield(
    lon: float,
    lat: float,
    max_distance_km: float,
    yield_results: dict | None = None,
    materials: list[str] | None = None,
    yield_map_dir: Path = YIELD_MAP_DIR,
    boundary_raster: Path = BOUNDARY_RASTER,
    zoom: float = MAP_ZOOM_FACTOR,
    figsize: tuple = (22, 11),
    save_path: str | None = None,
    dpi: int = 150,
):
    """
    Plot a 2×4 panel figure: pie chart + one map per material.

    Parameters
    ----------
    lon, lat            : origin coordinates (WGS84).
    max_distance_km     : catchment radius in km.
    yield_results       : optional pre-computed {material: tonnes} dict.
    materials           : list of material names (default: all 7).
    yield_map_dir       : directory containing the yield GeoTIFFs.
    boundary_raster     : path to the Europe land-mask GeoTIFF.
    zoom                : multiplier on catchment radius for map extent
                          (1.0 = tight, 1.8 = ~80 % extra context).
    figsize, save_path, dpi : figure settings.
    """
    if materials is None:
        materials = list(MATERIAL_COLORS.keys())
    assert len(materials) == 7, "This layout expects exactly 7 materials."

    max_dist_m = max_distance_km * 1000

    # Project origin → EPSG:3035
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)
    x_origin, y_origin = transformer.transform(lon, lat)

    # ── 1. Compute shared window (zoom × radius) ──────────────────────────────
    # Use the first available yield raster to derive window params
    ref_tif = next(
        (yield_map_dir / f"yield_tonnes_perYear_10km_{m}.tif"
         for m in materials
         if (yield_map_dir / f"yield_tonnes_perYear_10km_{m}.tif").exists()),
        None,
    )
    if ref_tif is None:
        raise FileNotFoundError("No yield rasters found in yield_map_dir.")

    row_min, row_max, col_min, col_max, pixel_size, base_transform = \
        _get_window_params(ref_tif, x_origin, y_origin, max_dist_m, zoom=zoom)

    # Pixel centre coordinates for the window
    x_grid, y_grid = _build_pixel_centres(
        base_transform, row_min, row_max, col_min, col_max)

    # Distance from origin for every pixel in the window
    dist_grid = np.sqrt((x_grid - x_origin) ** 2 + (y_grid - y_origin) ** 2)
    inside_mask = dist_grid <= max_dist_m

    # Map axis limits: always a square centred on the origin,
    # regardless of how much the raster window was clipped at the edges.
    half_extent = max_dist_m * zoom
    x_lim = (x_origin - half_extent, x_origin + half_extent)
    y_lim = (y_origin - half_extent, y_origin + half_extent)

    # ── 2. Load land mask ─────────────────────────────────────────────────────
    land_mask = _read_land_mask(boundary_raster, row_min, row_max, col_min, col_max)

    # ── 3. Load all yield layers ──────────────────────────────────────────────
    layers = {}
    totals = {}

    for mat in materials:
        tif_path = yield_map_dir / f"yield_tonnes_perYear_10km_{mat}.tif"
        if not tif_path.exists():
            print(f"[WARNING] Missing: {tif_path}")
            layers[mat] = None
            totals[mat] = 0.0
            continue

        data = _read_windowed(tif_path, row_min, row_max, col_min, col_max)
        layers[mat] = data

        valid_inside = np.where(inside_mask & ~np.isnan(data), data, np.nan)
        totals[mat] = (yield_results or {}).get(mat, float(np.nansum(valid_inside)))

    # ── 4. Build figure ───────────────────────────────────────────────────────
    fig = plt.figure(figsize=figsize, facecolor="white")
    fig.suptitle(
        f"Material yield within {max_distance_km:.0f} km of "
        f"({lat:.3f}°N, {lon:.3f}°E)",
        fontsize=15, fontweight="bold", y=0.98,
    )

    gs = fig.add_gridspec(
        2, 4,
        top=0.93, bottom=0.10,
        left=0.04, right=0.97,
        hspace=0.05, wspace=0.08,
    )
    axes = np.empty((2, 4), dtype=object)
    for r in range(2):
        for c in range(4):
            axes[r, c] = fig.add_subplot(gs[r, c])

    # ── 5. Pie chart ──────────────────────────────────────────────────────────
    ax_pie   = axes[0, 0]
    pie_vals  = [totals[m] for m in materials]
    pie_colors = [MATERIAL_COLORS[m] for m in materials]
    total_all  = sum(pie_vals)

    wedges, _, autotexts = ax_pie.pie(
        pie_vals, colors=pie_colors,
        autopct=lambda p: f"{p:.1f}%" if p >= 3 else "",
        pctdistance=0.75, startangle=90,
        wedgeprops=dict(linewidth=0.6, edgecolor="white"),
    )
    for at in autotexts:
        at.set_fontsize(7.5)
        at.set_color("white")
        at.set_fontweight("bold")

    ax_pie.set_title(f"Total: {total_all:,.0f} t/yr", fontsize=9, pad=6)
    ax_pie.set_aspect("equal")

    # ── 6. Material maps ──────────────────────────────────────────────────────
    grey_cmap = mcolors.LinearSegmentedColormap.from_list(
        "grey_yield", ["#d0d0d0", "#888888"], N=64)

    map_positions = [(0,1),(0,2),(0,3),(1,0),(1,1),(1,2),(1,3)]

    for mat, (ri, ci) in zip(materials, map_positions):
        ax    = axes[ri, ci]
        color = MATERIAL_COLORS[mat]
        cmap  = _make_sequential_cmap(color, mat)
        data  = layers[mat]

        _format_map_ax(ax, x_lim, y_lim)

        # --- land background (always, regardless of yield data) ---
        _draw_land(ax, x_grid, y_grid, land_mask)

        if data is None:
            _draw_circle_and_origin(ax, x_origin, y_origin, max_dist_m, color)
            ax.text(0.5, 0.46, "None within catchment",
                    ha="center", va="center", transform=ax.transAxes,
                    color="#444444", fontsize=8, style="italic")
            ax.set_title(f"{mat.capitalize()}  •  0 t/yr",
                         fontsize=8.5, fontweight="bold")
            continue

        all_valid = ~np.isnan(data)

        # --- outside-catchment yield pixels: grey ---
        outside_data = np.where(~inside_mask & all_valid, data, np.nan)
        if np.any(~np.isnan(outside_data)):
            ov = outside_data[~np.isnan(outside_data)]
            ov_min = max(ov.min(), 0.01)
            ov_max = max(ov.max(), ov_min * 1.001)
            ax.pcolormesh(
                x_grid, y_grid, outside_data,
                cmap=grey_cmap, shading="nearest",
                norm=mcolors.LogNorm(vmin=ov_min, vmax=ov_max),
                rasterized=True, zorder=2,
            )

        # --- inside-catchment yield pixels: material colour ---
        inside_data = np.where(inside_mask & all_valid, data, np.nan)
        valid_in    = inside_data[~np.isnan(inside_data)]

        if valid_in.size > 0:
            vmin = max(valid_in.min(), 0.01)
            vmax = max(valid_in.max(), vmin * 1.001)  # guard: vmax must exceed vmin
            norm = mcolors.LogNorm(vmin=vmin, vmax=vmax) if vmax > vmin \
                   else mcolors.Normalize(vmin=0, vmax=vmax)
            ax.pcolormesh(
                x_grid, y_grid, inside_data,
                cmap=cmap, shading="nearest", norm=norm,
                rasterized=True, zorder=3,
            )
        else:
            ax.text(0.5, 0.46, "None within catchment",
                    ha="center", va="center", transform=ax.transAxes,
                    color="#444444", fontsize=8, style="italic")

        _draw_circle_and_origin(ax, x_origin, y_origin, max_dist_m, color)
        ax.set_title(
            f"{mat.capitalize()}  •  {totals[mat]:,.0f} t/yr",
            fontsize=8.5, fontweight="bold",
        )

    # ── 7. Shared legend ──────────────────────────────────────────────────────
    legend_handles = [
        mpatches.Patch(facecolor=MATERIAL_COLORS[m], edgecolor="white",
                       linewidth=0.5, label=m.capitalize())
        for m in materials
    ]
    legend_handles += [
        mpatches.Patch(facecolor="#e0ddd8", edgecolor="#aaaaaa",
                       linewidth=0.5, label="Land (no yield data)"),
        mpatches.Patch(facecolor="#b0b0b0", edgecolor="none",
                       label="Outside catchment (yield)"),
        Line2D([0], [0], linestyle="--", color="#555555", linewidth=1.3,
               label=f"Catchment ({max_distance_km:.0f} km)"),
        Line2D([0], [0], marker="*", color="#555555", linestyle="None",
               markersize=9, label="Origin"),
    ]

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=len(legend_handles),
        bbox_to_anchor=(0.5, 0.01),
        fontsize=8.5,
        frameon=True, framealpha=0.9, edgecolor="#cccccc",
        handlelength=1.5,
    )

    # ── 8. Save / show ────────────────────────────────────────────────────────
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches=None, facecolor="white")
        print(f"Figure saved to: {save_path}")

    return fig, axes


# ── Example usage ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # from material_yield_estimator import estimate_yield
    # results = estimate_yield(lon=4.9, lat=52.37, max_distance_km=500, verbose=False)

    fig, axes = visualize_yield(
        lon=4.9,
        lat=52.37,
        max_distance_km=500,
        # yield_results=results,
        save_path="yield_map_Amsterdam_500km.png",
    )
    plt.show()

Figure saved to: yield_map_Amsterdam_500km.png


/var/folders/jy/n4b2f9d16xbckrmnhmlqjc1w0000gq/T/ipykernel_93816/2821213182.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Make gif

In [53]:
"""
generate_europe_gif.py
======================
Generates yield maps for N random locations across Europe,
saves each as a PNG, then compiles them into an animated GIF.
"""

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import imageio.v2 as imageio
from PIL import Image

# ── Settings ──────────────────────────────────────────────────────────────────

N_LOCATIONS        = 15
MAX_DISTANCE_KM    = 500
RANDOM_SEED        = 42
GIF_FRAME_DURATION = 1000   # milliseconds per frame

OUTPUT_DIR = Path("../../results/figures/europe_tour")
GIF_PATH   = Path("../../results/figures/europe_tour.gif")

LON_MIN, LON_MAX = -10.0, 30.0
LAT_MIN, LAT_MAX =  36.0, 65.0

NAMED_LOCATIONS = [
    ("Amsterdam",  4.90, 52.37),
    ("Madrid",    -3.70, 40.42),
    ("Warsaw",    21.01, 52.23),
    ("Stockholm", 18.07, 59.33),
    ("Rome",      12.50, 41.90),
]


def random_europe_locations(n: int, seed: int = RANDOM_SEED):
    rng = np.random.default_rng(seed)
    return [(rng.uniform(LON_MIN, LON_MAX), rng.uniform(LAT_MIN, LAT_MAX))
            for _ in range(n)]


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    n_random  = max(0, N_LOCATIONS - len(NAMED_LOCATIONS))
    locations = [(lon, lat, name) for name, lon, lat in NAMED_LOCATIONS] + \
                [(lon, lat, f"loc_{i+1:02d}")
                 for i, (lon, lat) in enumerate(random_europe_locations(n_random))]
    locations = locations[:N_LOCATIONS]

    png_paths = []

    for i, (lon, lat, label) in enumerate(locations):
        print(f"[{i+1}/{len(locations)}] {label}  ({lat:.2f}°N, {lon:.2f}°E)")
        save_path = OUTPUT_DIR / f"{i+1:02d}_{label.replace(' ', '_')}.png"
        try:
            results = estimate_yield(lon=lon, lat=lat,
                                     max_distance_km=MAX_DISTANCE_KM, verbose=False)
            fig, _ = visualize_yield(lon=lon, lat=lat,
                                     max_distance_km=MAX_DISTANCE_KM,
                                     yield_results=results,
                                     save_path=str(save_path), dpi=100)
            plt.close(fig)
            png_paths.append(save_path)
            print(f"    saved → {save_path.name}")
        except Exception as e:
            print(f"    [ERROR] {e}")

    if not png_paths:
        print("No images generated.")
        return

    print(f"\nCompiling {len(png_paths)} frames → {GIF_PATH}")
    GIF_PATH.parent.mkdir(parents=True, exist_ok=True)

    imgs = [Image.open(p).convert("RGB") for p in png_paths]
    target_size = imgs[0].size
    imgs = [img.resize(target_size, Image.LANCZOS) for img in imgs]

    imageio.mimsave(
        str(GIF_PATH),
        [np.array(img) for img in imgs],
        duration=GIF_FRAME_DURATION,
        loop=0,
    )
    print(f"Done! GIF saved to: {GIF_PATH}")


if __name__ == "__main__":
    main()

[1/15] Amsterdam  (52.37°N, 4.90°E)
Figure saved to: ../../results/figures/europe_tour/01_Amsterdam.png
    saved → 01_Amsterdam.png
[2/15] Madrid  (40.42°N, -3.70°E)
Figure saved to: ../../results/figures/europe_tour/02_Madrid.png
    saved → 02_Madrid.png
[3/15] Warsaw  (52.23°N, 21.01°E)
Figure saved to: ../../results/figures/europe_tour/03_Warsaw.png
    saved → 03_Warsaw.png
[4/15] Stockholm  (59.33°N, 18.07°E)
Figure saved to: ../../results/figures/europe_tour/04_Stockholm.png
    saved → 04_Stockholm.png
[5/15] Rome  (41.90°N, 12.50°E)
Figure saved to: ../../results/figures/europe_tour/05_Rome.png
    saved → 05_Rome.png
[6/15] loc_01  (48.73°N, 20.96°E)
Figure saved to: ../../results/figures/europe_tour/06_loc_01.png
    saved → 06_loc_01.png
[7/15] loc_02  (56.22°N, 24.34°E)
Figure saved to: ../../results/figures/europe_tour/07_loc_02.png
    saved → 07_loc_02.png
[8/15] loc_03  (64.29°N, -6.23°E)
Figure saved to: ../../results/figures/europe_tour/08_loc_03.png
    saved → 08_